Topic---Planner---Plan---Executor---Draft---(if not converged)---Critic---Feedback

In [ ]:
# self-correcting content generation pipeline — a multi-agent system that plans, writes, and critiques 
# blog posts in an iterative loop until the output reaches a quality threshold. 
"""
What you will build: -------
A Planner agent that produces a structured article outline from a topic
An Executor/Writer agent that turns the outline into a full blog post
A Critic agent that scores the article and provides actionable feedback
An orchestration loop in Python that coordinates these three agents, passing feedback from the Critic back to the Writer until the article is good enough (or a maximum number of iterations is reached)

What you will practice: -----
Creating multiple Agent objects with different models and output_type
Calling agents with Runner.run_sync() and working with structured Pydantic output
Writing imperative orchestration — a Python loop that controls agent execution, rather than using SDK handoffs or agent.as_tool()
Implementing convergence detection and bounded iteration
"""

In [ ]:
import mermaid as md
# Topic---Planner---Plan---Executor---Draft---(if not converged)---Critic---Feedback

md.Mermaid("""
flowchart LR
    A[Topic] --> B[Planner]
    B --> C[Plan]
    C --> D[Executor]
    D --> E[Draft]
    E --> F[Critic]
    F --> G[Feedback]
    G -- If not converged --> D
""")
# Every iterative agent loop must have a hard limit (MAX_ITERATIONS). Without it, a perfectionist critic could request 
# revisions forever. We also define a QUALITY_THRESHOLD — when the critic scores the article above this threshold, we stop early. 
# Together, these give you convergence detection with a safety net.

# Rule of thumb: Use the SDK's Agent + Runner.run_sync to simplify individual agent calls. Keep orchestration in Python 
# when the workflow is structured. Use handoffs/as_tool when the LLM needs to decide the flow (like triage routing).

In [ ]:
%pip install openai-agents pydantic mermaid-py nest-asyncio -q
from nest_asyncio import apply
apply()
POWERFUL_MODEL = 'o4-mini'
MODEL = 'gpt-5-mini'

In [ ]:
import getpass
import os
from pydantic import BaseModel, Field
from agents import Agent, Runner

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")

# Model configuration - use reasoning models for planning/critique, cheap model for writing
PLANNER_MODEL = POWERFUL_MODEL          # Reasoning model for high-quality planning
WRITER_MODEL = MODEL                 # Cost-effective model for content generation
CRITIC_MODEL = POWERFUL_MODEL           # Reasoning model for quality evaluation
MAX_ITERATIONS = 3                                          # Maximum revision cycles
QUALITY_THRESHOLD = 8                                       # Score out of 10 to stop iterating

In [ ]:
class ArticleCritique(BaseModel):
    """Structured critique from the reviewer."""
    score: int = Field(description="Quality score from 1 to 10")
    strengths: list[str] = Field(description="List of article strengths")
    weaknesses: list[str] = Field(description="List of article weaknesses")
    specific_feedback: str = Field(description="Detailed, actionable feedback for improvement")
    ready_to_publish: bool = Field(description="Whether the article is publication-ready")

In [ ]:
#----------SECTION 1 ------------------------
# Define the Three Agents
""" 
Create three Agent objects — one for each role. You already know how to do this from Lecture 80.
1. planner_agent — Uses PLANNER_MODEL. Instructions should tell it to produce a detailed blog post plan 
(title, intro hook, 3-5 sections with key points, code examples to include, conclusion). Returns raw text (no output_type).

2. writer_agent — Uses WRITER_MODEL. Instructions should tell it to write a complete, publication-ready blog post 
in markdown, 800-1200 words. Returns raw text (no output_type).

3. critic_agent — Uses CRITIC_MODEL. Instructions should tell it to evaluate the article on: plan adherence, technical accuracy, 
writing quality, practical value, and completeness. Uses output_type=ArticleCritique so the output is a parsed Pydantic instance.

"""
planner_agent = Agent(
    name="Article Planner",
    instructions="""You are a senior content strategist. Create a detailed blog post plan.

Your plan should include:
1. A compelling title
2. An introduction hook
3. 3-5 main sections with key points for each
4. Code examples or practical demonstrations to include
5. A conclusion with key takeaways

Return the plan as clear, structured text. This plan will be given to a writer to produce the full article.""",
    model=PLANNER_MODEL,
)

writer_agent = Agent(
    name="Article Writer",
    instructions="""You are a skilled technical writer. Write a complete, publication-ready blog post.

Requirements:
- Write in a clear, engaging style suitable for a technical blog
- Include code examples where the plan specifies them
- Use markdown formatting (headers, code blocks, bullet points)
- Aim for 800-1200 words
- Make it practical and actionable""",
    model=WRITER_MODEL,
)

critic_agent = Agent(
    name="Article Critic",
    instructions="""I want you to be extremely critical. You are a senior technical editor reviewing a blog post.

Evaluate the article on these criteria:
1. Adherence to the plan (does it cover all planned sections?)
2. Technical accuracy
3. Writing quality (clarity, engagement, flow)
4. Practical value (actionable takeaways, useful code examples)
5. Completeness (introduction, body, conclusion)

Be rigorous but fair. Only give a score of 8+ if the article is genuinely publication-ready.""",
    model=CRITIC_MODEL,
    output_type=ArticleCritique,
)

print(f"Planner: {planner_agent.name} ({planner_agent.model})")
print(f"Writer:  {writer_agent.name} ({writer_agent.model})")
print(f"Critic:  {critic_agent.name} ({critic_agent.model}) -> output_type={critic_agent.output_type.__name__}")

In [ ]:
# Section 2: Write the Helper Functions
def plan_article(topic, audience="general technical audience"):
    """Use a reasoning model to generate a detailed article plan."""
    result = Runner.run_sync(                                           # Run planner_agent with this prompt, wait until it finishes (run_sync), and give me the result."
                planner_agent,
                f"""Create a detailed blog post plan for the following topic.
                Topic: {topic}
                Target Audience: {audience}"""
                # This is similar to what we had done in earlier programs 
                #    planner_agent = Agent(
                #    name="Article Planner",
                #    instructions="Create detailed article plans...",
                #    model="gpt-5-mini"
        )
    print("=" * 60)
    print("ARTICLE PLAN")
    print("=" * 60)
    print(result.final_output)
    return result.final_output


def write_article(plan, feedback=None):
    """Write the full article from a plan, optionally incorporating critic feedback."""
    prompt = f"""Write a complete, publication-ready blog post based on this plan:

{plan}"""

    if feedback:
        prompt += f"""

IMPORTANT: The following feedback was provided by a reviewer. Address ALL points:

{feedback}"""

    result = Runner.run_sync(writer_agent, prompt)

    print(f"\nArticle generated ({len(result.final_output.split())} words)")
    return result.final_output


def critique_article(article, plan, iteration):
    """Evaluate the article and return structured feedback as an ArticleCritique."""
    result = Runner.run_sync(
        critic_agent,
        f"""ORIGINAL PLAN:
{plan}

ARTICLE (Iteration {iteration}):
{article}"""
    )

    critique = result.final_output

    print(f"\n{'='*60}")
    print(f"CRITIQUE (Iteration {iteration})")
    print(f"{'='*60}")
    print(f"Score: {critique.score}/10")
    print(f"Ready to publish: {critique.ready_to_publish}")
    print(f"Strengths:")
    for s in critique.strengths:
        print(f"  + {s}")
    print(f"Weaknesses:")
    for w in critique.weaknesses:
        print(f"  - {w}")
    print(f"\nFeedback: {critique.specific_feedback}")

    return critique